# Rapport 

##### Objectif : créer deux bdd (customer et orders) et y appliquer 3 requêtes à savoir :
###### Quels sont les 10 clients les plus actifs (plus de commandes) ?

###### Quel est le chiffre d’affaires total par pays ?

###### Quels clients n’ont jamais passé de commande ? 

### Import des bibliothèques 

In [16]:
import sqlite3 
from random import randint,choice,uniform
from datetime import datetime, timedelta

### Création de la connexion et du cursor

###### note : création du cursor pour pour lire les lignes de code et d'une bde SQL en mémoire temporaire pour plus de simplicité

In [17]:
conn = sqlite3.connect(":memory:")
cursor=conn.cursor()

### Création des bdd 

In [18]:
conn.executescript(""" 
CREATE TABLE customers(
id INTEGER PRIMARY KEY,
name TEXT,
country TEXT
);
CREATE TABLE orders(
order_id INTEGER PRIMARY KEY,
customer_id INTEGER,
order_date DATETIME,
amount REAL
);
""")

### Input des données dans les tables 

In [19]:
cursor.executescript("""
DELETE FROM customers
;
DELETE FROM orders
; """)

###### Note : On va randomiser les rentrées de données 

In [25]:
names = ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank", "Grace", "Hector", "Ivy", "Jack"]
countries = ["France", "USA", "Germany", "UK", "Spain"]
order_date_str = order_date.strftime("%Y-%m-%d %H:%M:%S")

# insertion dans la bdd customers 
for _ in range(10):  # 10 clients
    name = choice(names)
    country = choice(countries)
    cursor.execute(
        "INSERT INTO customers (name, country) VALUES (?, ?)",  
        (name, country)
    )

# insertion orders
cursor.execute("SELECT id FROM customers")
customer_ids = [row[0] for row in cursor.fetchall()]  # Récupérer les IDs générés

order_id = 1
for customer_id in customer_ids:
    # entre 0 et 5 commandes par clients 
    for _ in range(randint(0, 5)):
        order_date = datetime.now() - timedelta(days=randint(0, 90))
        # Conversion format date
        order_date_str = order_date.strftime("%Y-%m-%d %H:%M:%S")  
        amount = round(uniform(10, 500), 2)
        cursor.execute(
            "INSERT INTO orders (customer_id, order_date, amount) VALUES (?, ?, ?)",
            (customer_id, order_date_str, amount)
        )
        order_id += 1
conn.commit()

###### Note : on peut visualiser les tables avant d'entamer quoi que ce soit pour vérifier 

In [26]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables=cursor.fetchall()
print(tables)

[('customers',), ('orders',)]


In [27]:
cursor.execute("PRAGMA table_info(customers)")
print(cursor.fetchall())

[(0, 'id', 'INTEGER', 0, None, 1), (1, 'name', 'TEXT', 0, None, 0), (2, 'country', 'TEXT', 0, None, 0)]


### 1ère tache : les 10 clients les plus actifs 

In [31]:
cursor.execute("""
SELECT c.name, c.id, COUNT(o.order_id)
FROM customers c 
JOIN orders o
ON c.id=o.customer_id
GROUP BY c.id,c.name
ORDER BY COUNT(o.order_id) DESC
LIMIT 10
""")

### Affichage des colonnes 

column=[desc[0] for desc in cursor.description] ## desc[0] est le nom de la colonne
print('\t'.join(column))  ## sépare chaque nom de colonne par une fois tab
for row in rows:
    print('\t'.join(str(x) for x in row))
    

name	id	COUNT(o.order_id)
Grace	2	13
Ivy	10	10
Alice	4	9
Alice	9	9
Hector	5	8
Alice	7	8
Ivy	11	8
Eve	18	8
Charlie	21	8
Frank	27	8


### Tâche 2 : chiffre d'affaire total par pays 

In [38]:
cursor.execute("""
SELECT c.country, SUM(o.amount)
FROM customers c
JOIN orders o
ON c.id=o.customer_id
GROUP BY c.country
""")

rows=cursor.fetchall()
column=[desc[0] for desc in cursor.description]
print('\t'.join(column))
for row in rows:
    print('\t'.join(str(x) for x in row), '€')

country	SUM(o.amount)
France	8501.04 €
Germany	9852.87 €
Spain	8004.37 €
UK	12455.26 €
USA	10275.35 €


### Tâche 3 : clients qui n'ont jamais passé commande 

In [46]:
cursor.execute("""
SELECT c.name,c.id 
FROM customers c 
LEFT JOIN orders o 
ON c.id=o.customer_id
WHERE o.order_id IS NULL

""")

rows=cursor.fetchall()
column=[desc[0] for desc in cursor.description]
print('\t'.join(column))
for row in rows:
    print('\t'.join(str(x) for x in row))

name	id
Diana	13
Bob	33


###### Note : on pourrait faire une fonction afficher résultat pour éviter de réecrire l'affichage a chaque requête dans une quete d'optimisation 

### Fin du rapport et fermeture de la connexion avec la bdd

In [47]:
conn.close()

###### note : vérification de la fermeture 

In [48]:
cursor.execute("SELECT c.name FROM customers")

ProgrammingError: Cannot operate on a closed database.